# 3.4 找到位置後取回什麼？

# 第 3 章：Attention，自己決定看哪裡

前置：向量、矩陣乘法和 softmax。先用單 head；每個小節都畫得出矩陣。完整模型仍留到第4章。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：問路譬喻：想找什麼、誰知道、拿回什麼**

Q/K 決定讀哪裡；V 是讀回的內容。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/qkv.svg")))

**先想一想：**key 兩維、value 三維，輸出是兩維嗎？

找到位置後要拿回的內容是 Value。K 可以只描述索引線索，V 保存真正要混合的資料，兩者的維度也可不同。

**把直覺寫成數學：**$softmax(QK^T)V$ 的 shape 是 $[T_q,D_v]$。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
q = torch.tensor([[1.0, 0.0]])
k = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
v = torch.tensor([[10.0, 20.0, 30.0], [-1.0, -2.0, -3.0]])
weights = (q @ k.T).softmax(-1)
print(weights, weights @ v)

**如何讀結果：**Q/K 決定讀哪裡，V 決定讀回什麼；輸出 feature 數由 V 決定。

**只改一件事：**只更改 V，檢查權重不變。
